[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S19_ml_evaluacion.ipynb)

# Sesión 19 · Evaluar un clasificador

**Módulo 5: Machine Learning** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Explicar por qué la accuracy engaña cuando una clase es muy rara.
2. Leer una matriz de confusión y calcular precision, recall y F1.
3. Evaluar las probabilidades de un modelo con la curva ROC y el AUC.
4. Estimar el desempeño con validación cruzada y elegir un umbral según los costos del negocio.

## 📋 Qué debes saber antes
Sesión 18: clasificadores, `predict_proba` y umbral de decisión.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`, con los nombres de variables que se piden.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos, aplica el estilo de gráficos y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, aplica el estilo de gráficos y carga los verificadores.
import copy
import hashlib
import math
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})

# ---------- Datos de práctica: transacciones con tarjeta, algunas fraudulentas ----------
_n = 4000
_monto = np.round(rng.lognormal(4.5, 1.0, _n), 2)
_noct = (rng.random(_n) < 0.15).astype(int)
_inter = (rng.random(_n) < 0.1).astype(int)
_intentos = rng.poisson(0.2, _n)
_dias = np.round(rng.exponential(8, _n), 1)
_logit = -5.6 + 0.004 * _monto + 1.6 * _noct + 2.2 * _inter + 1.2 * _intentos - 0.05 * _dias
transacciones = pd.DataFrame({
    "monto": _monto, "es_nocturna": _noct, "es_internacional": _inter, "intentos_fallidos": _intentos,
    "dias_ultima_compra": _dias, "fraude": (rng.random(_n) < 1 / (1 + np.exp(-_logit))).astype(int),
})
VARIABLES = ["monto", "es_nocturna", "es_internacional", "intentos_fallidos", "dias_ultima_compra"]
COSTO_FP = 50      # soles: revisar una transacción sana que se marcó como fraude
COSTO_FN = 400     # soles: pérdida promedio por un fraude que no se detectó

_D = copy.deepcopy({"transacciones": transacciones})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")

def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _datos():
    xt, yt = globals().get("X_test"), globals().get("y_test")
    if isinstance(xt, pd.DataFrame) and isinstance(yt, pd.Series) and len(xt) == len(yt):
        return xt, [int(v) for v in yt.tolist()]
    return None


def _vector(r, nombre, largo):
    v = r.var(nombre)
    if v is _FALTA:
        return None
    try:
        arr = [float(x) for x in np.asarray(v, dtype=float).ravel()]
    except (TypeError, ValueError):
        arr = None
    if arr is None or len(arr) != largo:
        r.mal(f"`{nombre}` debería tener {largo} valores, uno por fila de prueba.")
        return None
    return arr


def _conteos(real, pred):
    """Verdaderos negativos, falsos positivos, falsos negativos y verdaderos positivos, contados a mano."""
    vn = sum(1 for a, b in zip(real, pred) if a == 0 and b == 0)
    fp = sum(1 for a, b in zip(real, pred) if a == 0 and b == 1)
    fn = sum(1 for a, b in zip(real, pred) if a == 1 and b == 0)
    vp = sum(1 for a, b in zip(real, pred) if a == 1 and b == 1)
    return vn, fp, fn, vp


def _auc(real, prob):
    """AUC con la fórmula de rangos de Mann-Whitney (sin scikit-learn); los empates reciben el rango promedio."""
    orden = sorted(range(len(prob)), key=lambda i: prob[i])
    rangos = [0.0] * len(prob)
    i = 0
    while i < len(orden):
        j = i
        while j + 1 < len(orden) and prob[orden[j + 1]] == prob[orden[i]]:
            j += 1
        for k in range(i, j + 1):
            rangos[orden[k]] = (i + j) / 2 + 1
        i = j + 1
    pos = [rangos[i] for i in range(len(real)) if real[i] == 1]
    n_pos, n_neg = len(pos), len(real) - len(pos)
    return (math.fsum(pos) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def _pred_de(nombre_modelo, xt):
    m = globals().get(nombre_modelo)
    return [float(v) for v in m.predict(xt)] if m is not None and hasattr(m, "classes_") else None


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    t = _D["transacciones"]
    _esc(r, "pct_fraude", round(statistics.fmean(t["fraude"].tolist()) * 100, 2), "el porcentaje de transacciones fraudulentas, con 2 decimales", tol=0.011)
    d = _datos()
    if d is None:
        r.mal("Faltan `X_test` e `y_test`.")
    else:
        xt, yt = d
        xs = globals().get("X_train")
        if len(xt) != 1200 or not isinstance(xs, pd.DataFrame) or len(xs) != 2800 or [str(c) for c in xt.columns] != VARIABLES:
            r.mal("La partición debería dejar 30 % para prueba (1200 filas) con las columnas de `VARIABLES`.")
        elif abs(statistics.fmean(yt) - statistics.fmean(t["fraude"].tolist())) > 0.002:
            r.mal("La proporción de fraudes en prueba no coincide con la del total: usa `stratify=y`.")
        else:
            r.ok("La partición es estratificada.")
        for nombre, tipo, acc in (("dummy", "DummyClassifier", "acc_tonto"), ("modelo_log", "LogisticRegression", "acc_log")):
            m = r.var(nombre)
            if m is _FALTA:
                continue
            if type(m).__name__ != tipo or not hasattr(m, "classes_"):
                r.mal(f"`{nombre}` debería ser un `{tipo}` entrenado.")
                continue
            pred = [float(v) for v in m.predict(xt)]
            if nombre == "dummy" and set(pred) != {0.0}:
                r.mal("`dummy` debería predecir siempre la clase más frecuente (`strategy=\"most_frequent\"`).")
            _esc(r, acc, sum(1 for a, b in zip(yt, pred) if a == b) / len(yt), f"la accuracy de `{nombre}` en prueba", tol=1e-9)
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_tonto_detecta": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    d = _datos()
    if d is None:
        r.mal("Primero resuelve el ejercicio 1.")
    else:
        xt, yt = d
        pred = _pred_de("modelo_log", xt)
        if pred is None:
            r.mal("Primero entrena `modelo_log`.")
        else:
            vn, fp, fn, vp = _conteos(yt, pred)
            cm = r.var("cm")
            if cm is not _FALTA:
                cm = np.asarray(cm)
                if cm.shape != (2, 2):
                    r.mal("`cm` debería ser una matriz de 2 × 2.")
                elif cm.tolist() == [[vn, fp], [fn, vp]]:
                    r.ok("`cm` es la matriz de confusión del modelo en prueba.")
                elif cm.tolist() == [[vn, fn], [fp, vp]]:
                    r.mal("`cm` está transpuesta: el primer argumento de `confusion_matrix` son los valores reales.")
                else:
                    r.mal("`cm` debería ser `confusion_matrix(y_test, pred_log)`.")
            for nombre, valor in (("vn", vn), ("fp", fp), ("fn", fn), ("vp", vp)):
                _esc(r, nombre, valor, "desempaqueta `cm.ravel()` en el orden vn, fp, fn, vp")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_filas": "199fc66fd524466f5dd285f7da2a168b4af9fd27009fdfc4c6a4b8b7ad931f6a",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    d = _datos()
    if d is None:
        r.mal("Primero resuelve el ejercicio 1.")
    else:
        xt, yt = d
        pred = _pred_de("modelo_log", xt)
        tonto = _pred_de("dummy", xt)
        if pred is None or tonto is None:
            r.mal("Primero entrena `modelo_log` y `dummy`.")
        else:
            vn, fp, fn, vp = _conteos(yt, pred)
            p = vp / (vp + fp) if vp + fp else 0.0
            rc = vp / (vp + fn)
            f1 = 2 * p * rc / (p + rc) if p + rc else 0.0
            for nombre, valor in (("precision", p), ("recall", rc), ("f1", f1), ("precision_sk", p), ("recall_sk", rc), ("f1_sk", f1)):
                _esc(r, nombre, valor, "revisa la fórmula con los conteos de la matriz de confusión", tol=1e-9)
            _esc(r, "recall_tonto", 0.0, "el recall del modelo que nunca dice fraude")
            _esc(r, "precision_tonto", 0.0, "la precision del modelo que nunca dice fraude, con `zero_division=0`")
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_metrica_detectar": "aeda36e376463f585d3d9d71f3f4542736e81772cb274e0326ba752258b5ea67",
        "pred_metrica_alarmas": "874051f777ae46eb7803afe40f4f87a10c79ed9d69f598c412896f9d9b1869ec",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    d = _datos()
    m = globals().get("modelo_log")
    if d is None or m is None or not hasattr(m, "classes_"):
        r.mal("Primero resuelve el ejercicio 1.")
    else:
        xt, yt = d
        ref = [float(v) for v in m.predict_proba(xt)[:, 1]]
        prob = _vector(r, "prob_log", len(yt))
        if prob is not None:
            r.ok("`prob_log` es correcto.") if _cerca_lista(prob, ref, 1e-12) else r.mal("`prob_log` debería ser la probabilidad de fraude (columna 1 de `predict_proba`) en prueba.")
        auc = _auc(yt, ref)
        _esc(r, "auc_log", auc, "el área bajo la curva ROC con las probabilidades (no con las clases predichas)", tol=1e-9)
        _esc(r, "auc_tonto", 0.5, "el AUC de las probabilidades del modelo tonto", tol=1e-9)
        ax = _grafico(r, "ax_roc")
        if ax is not None:
            curvas = [l for l in ax.get_lines() if len(l.get_xdata()) > 2]
            diagonal = [l for l in ax.get_lines() if len(l.get_xdata()) == 2 and _cerca_lista(l.get_xdata(), l.get_ydata())]
            bien = False
            for l in curvas:
                x, y = [float(v) for v in l.get_xdata()], [float(v) for v in l.get_ydata()]
                area = math.fsum((x[i + 1] - x[i]) * (y[i + 1] + y[i]) / 2 for i in range(len(x) - 1))
                if x[0] == 0 and y[0] == 0 and x[-1] == 1 and y[-1] == 1 and abs(area - auc) < 1e-6:
                    bien = True
            if not bien:
                r.mal("`ax_roc` debería tener la curva ROC de `prob_log` (tasa de falsos positivos en x y de verdaderos positivos en y).")
            elif not diagonal:
                r.mal("Agrega la diagonal del azar, de (0, 0) a (1, 1).")
            elif ax.get_legend() is None:
                r.mal("Agrega una leyenda que diga qué es cada línea.")
            else:
                r.ok("`ax_roc` muestra la curva ROC y la línea del azar.")
            _rotulos(r, "ax_roc", ax, None, "Tasa de falsos positivos", "Tasa de verdaderos positivos (recall)")
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_auc_azar": "944b434bc2b311751e00501f0831f9072811f8675094dcd161bb4f23eb67d567",
    })
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    for nombre, media, desv in (("auc_cv", "auc_cv_media", "auc_cv_desv"),):
        v = r.var(nombre)
        if v is _FALTA:
            continue
        arr = np.asarray(v, dtype=float).ravel()
        if len(arr) != 5 or not ((arr >= 0) & (arr <= 1)).all():
            r.mal(f"`{nombre}` debería tener 5 valores entre 0 y 1: uno por pliegue.")
            continue
        if arr.mean() < 0.7:
            r.mal(f"Los AUC de `{nombre}` son bajos: ¿usaste `scoring=\"roc_auc\"` y una regresión logística?")
            continue
        r.ok(f"`{nombre}` tiene un AUC por pliegue.")
        _esc(r, media, statistics.fmean(arr.tolist()), f"el promedio de `{nombre}`", tol=1e-9)
        _esc(r, desv, statistics.pstdev(arr.tolist()), f"la desviación estándar de `{nombre}` (la de NumPy)", tol=1e-9)
    v = r.var("recall_cv_media")
    if v is not _FALTA:
        auc_m = globals().get("auc_cv_media")
        if not _es_numero(v) or not 0 <= float(v) <= 1:
            r.mal("`recall_cv_media` debería ser el promedio de un recall por pliegue, entre 0 y 1.")
        elif _es_numero(auc_m) and abs(float(v) - float(auc_m)) < 1e-9:
            r.mal("`recall_cv_media` es igual al AUC: cambia `scoring` a `\"recall\"`.")
        else:
            r.ok("`recall_cv_media` es un recall promedio de validación cruzada.")
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_n_entrenamientos": "345e42ad061bc48d0359a369b1c6c567a1c82108b53e556c40e79578d7443105",
    })
    r.fin()


def _costos_ref(prob, real):
    salida = {}
    for u in [round(0.05 * k, 2) for k in range(1, 20)]:
        pred = [1 if p >= u else 0 for p in prob]
        _, fp, fn, _ = _conteos(real, pred)
        salida[u] = COSTO_FP * fp + COSTO_FN * fn
    return salida


def check_reto():
    r = _Revision("Reto final")
    d = _datos()
    prob = globals().get("prob_log")
    if d is None or prob is None:
        r.mal("Primero resuelve los ejercicios 1 y 4.")
    else:
        yt = d[1]
        costos = _costos_ref([float(p) for p in np.asarray(prob).ravel()], yt)
        claves = list(costos)
        _ser(r, "costos", [costos[k] for k in claves], "el costo total (falsos positivos por `COSTO_FP` más falsos negativos por `COSTO_FN`) de cada umbral, de 0.05 a 0.95",
             indice=claves, tol=1e-6)
        mejor = min(claves, key=lambda k: (costos[k], k))
        _esc(r, "umbral_optimo", mejor, "el umbral de menor costo (si hay empate, el menor)", tol=1e-9)
        _esc(r, "ahorro", costos[0.5] - costos[mejor], "el costo con umbral 0.5 menos el costo con el umbral óptimo", tol=1e-6)
        ax = _grafico(r, "ax_costos")
        if ax is not None:
            lineas = [l for l in ax.get_lines() if len(l.get_ydata()) == len(claves)]
            verticales = [l for l in ax.get_lines() if len(set(map(float, l.get_xdata()))) == 1]
            if not lineas or not _cerca_lista(lineas[0].get_ydata(), [costos[k] for k in claves], 1e-6):
                r.mal("`ax_costos` debería tener una línea con el costo de cada umbral.")
            elif not any(abs(float(l.get_xdata()[0]) - mejor) < 1e-9 for l in verticales):
                r.mal("Marca el umbral óptimo con una línea vertical.")
            else:
                r.ok("`ax_costos` muestra la curva de costo y el umbral óptimo.")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    d = _datos()
    m = r.var("modelo_bal")
    if d is not None and m is not _FALTA:
        xt, yt = d
        if type(m).__name__ != "LogisticRegression" or getattr(m, "class_weight", None) != "balanced" or not hasattr(m, "classes_"):
            r.mal("`modelo_bal` debería ser una regresión logística entrenada con `class_weight=\"balanced\"`.")
        else:
            vn, fp, fn, vp = _conteos(yt, [float(v) for v in m.predict(xt)])
            _esc(r, "recall_bal", vp / (vp + fn), "el recall en prueba de `modelo_bal`", tol=1e-9)
            _esc(r, "precision_bal", vp / (vp + fp) if vp + fp else 0.0, "la precision en prueba de `modelo_bal`", tol=1e-9)
    r.fin()


print("✅ Setup listo. Datos generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
`transacciones`: 4000 compras con tarjeta, con su monto, si fue de noche, si fue internacional, los intentos fallidos previos, los días desde la última compra y si fue **fraude** (1) o no (0). `VARIABLES` tiene los nombres de las cinco variables. `COSTO_FP` y `COSTO_FN` son lo que le cuesta al banco cada tipo de error.

In [ ]:
print(transacciones.head(), "\n")
print(transacciones["fraude"].value_counts(), "\n")
print(transacciones.groupby("fraude")[VARIABLES].mean().round(2))

---
## 1. La trampa de la accuracy

### 📘 Concepto
La **accuracy** es la proporción de aciertos. Suena bien, pero engaña cuando una clase es rara: si solo el 3 % de las transacciones es fraude, un modelo que dice **siempre** "no es fraude" acierta el 97 % de las veces... y no detecta ni un solo fraude.

Para desenmascararlo, compara siempre con un modelo tonto: `DummyClassifier(strategy="most_frequent")` predice siempre la clase más común. Es el baseline de la clasificación, como el promedio lo fue en la regresión.

In [ ]:
from sklearn.dummy import DummyClassifier

y_ej = pd.Series([0] * 97 + [1] * 3)
X_ej = pd.DataFrame({"x": range(100)})
tonto_ej = DummyClassifier(strategy="most_frequent").fit(X_ej, y_ej)
print((tonto_ej.predict(X_ej) == y_ej).mean(), tonto_ej.predict(X_ej).sum())

### ✍️ Tu turno · Ejercicio 1: dos modelos con "buena" accuracy
**Parte A.**
1. `pct_fraude`: el porcentaje de transacciones fraudulentas, con 2 decimales.
2. `X` (columnas de `VARIABLES`) e `y` (`fraude`), y `X_train`, `X_test`, `y_train`, `y_test` con 30 % para prueba, `random_state=42` y estratificado.
3. `dummy`: un `DummyClassifier` que predice la clase más frecuente; `pred_tonto` y `acc_tonto`: sus predicciones y su accuracy en prueba.
4. `modelo_log`: una regresión logística (`max_iter=1000`); `pred_log` y `acc_log`: sus predicciones y su accuracy en prueba.

**Parte B.** Responde en `pred_tonto_detecta` con `"sí"` o `"no"`: ¿el modelo tonto detecta algún fraude?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Es el flujo de la sesión 18. El `DummyClassifier` también se entrena con `fit`, aunque solo "aprende" cuál es la clase más común.
</details>

<details><summary>💡 Pista 2</summary>

`dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)`. La accuracy es `(pred == y_test).mean()`.
</details>

---
## 2. La matriz de confusión

### 📘 Concepto
La **matriz de confusión** cuenta los cuatro resultados posibles:

| | predice 0 | predice 1 |
|---|---|---|
| **real 0** | verdaderos negativos (VN) | falsos positivos (FP): falsa alarma |
| **real 1** | falsos negativos (FN): fraude que se escapa | verdaderos positivos (VP) |

`confusion_matrix(y_real, y_predicho)` la devuelve con las **filas como valores reales** y las columnas como predichos. `cm.ravel()` la aplana en el orden VN, FP, FN, VP.

In [ ]:
from sklearn.metrics import confusion_matrix

real_ej = [0, 0, 0, 1, 1, 0, 1, 0]
pred_ej = [0, 1, 0, 1, 0, 0, 1, 0]
cm_ej = confusion_matrix(real_ej, pred_ej)
print(cm_ej)
print(cm_ej.ravel())       # vn, fp, fn, vp

### ✍️ Tu turno · Ejercicio 2: ¿en qué se equivoca?
**Parte A.**
1. `cm`: la matriz de confusión de `modelo_log` en prueba.
2. `vn`, `fp`, `fn`, `vp`: sus cuatro valores, desempaquetados en una línea.

¿Cuántos fraudes se le escapan al modelo? ¿Cuántas falsas alarmas da?

**Parte B.** Responde en `pred_filas` con `"reales"` o `"predichos"`: ¿qué representan las filas de `confusion_matrix`?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

El orden de los argumentos importa: primero los valores reales.
</details>

<details><summary>💡 Pista 2</summary>

`vn, fp, fn, vp = cm.ravel()`.
</details>

---
## 3. Precision, recall y F1

### 📘 Concepto
| Métrica | Fórmula | Responde |
|---|---|---|
| **Precision** | VP / (VP + FP) | De lo que marqué como fraude, ¿cuánto era fraude? (pocas falsas alarmas) |
| **Recall** | VP / (VP + FN) | De los fraudes reales, ¿cuántos detecté? (pocos casos escapados) |
| **F1** | 2 · P · R / (P + R) | Un solo número que equilibra las dos |

Suelen moverse en direcciones opuestas: marcar más casos sube el recall y baja la precision. Cuál importa más depende del negocio.

En `sklearn.metrics` están `precision_score`, `recall_score` y `f1_score`. Si el modelo nunca predice 1, la precision es 0 / 0: con `zero_division=0` se define como 0 y no aparece un aviso.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

print(precision_score(real_ej, pred_ej), recall_score(real_ej, pred_ej), f1_score(real_ej, pred_ej))
print(precision_score(real_ej, [0] * 8, zero_division=0))

### ✍️ Tu turno · Ejercicio 3: métricas que sí miran los fraudes
**Parte A.**
1. `precision`, `recall` y `f1` de `modelo_log`, calculados **a mano** con `vn`, `fp`, `fn` y `vp`.
2. `precision_sk`, `recall_sk` y `f1_sk`: los mismos, con las funciones de scikit-learn (deben coincidir).
3. `recall_tonto` y `precision_tonto`: las del modelo tonto (`zero_division=0`).

Compara con la accuracy: ¿qué cuenta cada número sobre los dos modelos?

**Parte B.** Responde con `"precision"` o `"recall"`:

| Variable | Pregunta |
|---|---|
| `pred_metrica_detectar` | ¿qué métrica mide cuántos de los fraudes reales se detectan? |
| `pred_metrica_alarmas` | ¿qué métrica baja cuando hay muchas falsas alarmas? |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Usa las fórmulas de la tabla con los cuatro conteos.
</details>

<details><summary>💡 Pista 2</summary>

Para las versiones de scikit-learn, el orden es `(y_test, pred_log)`. Para el modelo tonto, usa `pred_tonto`.
</details>

---
## 4. Curva ROC y AUC

### 📘 Concepto
Precision y recall dependen del umbral. La **curva ROC** evalúa las probabilidades con **todos** los umbrales a la vez: para cada umbral, dibuja la tasa de falsos positivos (x) contra la tasa de verdaderos positivos, que es el recall (y).

- Un modelo al azar queda sobre la diagonal.
- Cuanto más se acerca la curva a la esquina superior izquierda, mejor.
- El **AUC** (área bajo la curva) lo resume en un número: 0.5 es azar y 1 es perfecto. Equivale a la probabilidad de que un fraude al azar reciba mayor puntaje que una transacción sana al azar.

`roc_auc_score(y_real, probabilidades)` calcula el AUC y `roc_curve(y_real, probabilidades)` devuelve los puntos de la curva (`fpr`, `tpr` y los umbrales). Ojo: se calculan con **probabilidades**, no con las clases predichas.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

prob_ej = [0.1, 0.4, 0.35, 0.8, 0.3, 0.2, 0.9, 0.05]
print(roc_auc_score(real_ej, prob_ej))
fpr_ej, tpr_ej, _ = roc_curve(real_ej, prob_ej)
print(fpr_ej, tpr_ej)

### ✍️ Tu turno · Ejercicio 4: más allá del umbral
**Parte A.**
1. `prob_log`: la probabilidad de fraude de `modelo_log` en prueba, y `auc_log`: su AUC.
2. `auc_tonto`: el AUC de las probabilidades del modelo tonto.
3. `fig_roc, ax_roc`: la curva ROC de `modelo_log` (con `label` que muestre su AUC con 2 decimales) y la diagonal del azar en `GRIS` con `label="azar"`, con leyenda, eje x `Tasa de falsos positivos` y eje y `Tasa de verdaderos positivos (recall)`.

**Parte B.** Predice **sin ejecutar**: `pred_auc_azar` = el AUC de un modelo que asigna probabilidades al azar.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

`roc_curve` devuelve tres arrays; para el gráfico usa los dos primeros.
</details>

<details><summary>💡 Pista 2</summary>

`ax_roc.plot(fpr, tpr, label=f"logística (AUC = {auc_log:.2f})")` y `ax_roc.plot([0, 1], [0, 1], color=GRIS, label="azar")`.
</details>

---
## 5. Validación cruzada

### 📘 Concepto
Una sola partición entrenamiento/prueba puede salir con suerte o sin ella, sobre todo con pocos fraudes. La **validación cruzada** repite la evaluación varias veces:
1. Divide los datos en `k` partes (*pliegues*).
2. Entrena con `k - 1` partes y evalúa con la restante.
3. Repite hasta que cada parte fue prueba una vez: `k` entrenamientos y `k` notas.

El promedio estima el desempeño esperado y la desviación dice qué tan estable es. Con clases desbalanceadas, usa `StratifiedKFold` para que cada pliegue tenga la misma proporción de fraudes:

```python
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
notas = cross_val_score(modelo, X, y, cv=cv, scoring="roc_auc")
```

`cross_val_score` entrena modelos nuevos por su cuenta: se le pasa un modelo sin entrenar y todos los datos.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

X_cv_ej = pd.DataFrame({"x": np.r_[np.arange(40), np.arange(10) + 30]})
y_cv_ej = pd.Series([0] * 40 + [1] * 10)
cv_ej = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print(cross_val_score(LogisticRegression(max_iter=1000), X_cv_ej, y_cv_ej, cv=cv_ej, scoring="roc_auc").round(3))

### ✍️ Tu turno · Ejercicio 5: una nota más confiable
**Parte A.**
1. `cv`: un `StratifiedKFold` de 5 pliegues, mezclado, con `random_state=42`.
2. `auc_cv`: el AUC de una regresión logística nueva (`max_iter=1000`) en cada pliegue, usando `X` e `y` completos; `auc_cv_media` y `auc_cv_desv`: su promedio y su desviación estándar.
3. `recall_cv_media`: el recall promedio de validación cruzada del mismo modelo.

¿Es estable el AUC entre pliegues? ¿Y el recall?

**Parte B.** Predice **sin ejecutar**: `pred_n_entrenamientos` = cuántos modelos entrena `cross_val_score` con `cv` de 5 pliegues.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

`cross_val_score` devuelve un array con una nota por pliegue; usa `.mean()` y `.std()`.
</details>

<details><summary>💡 Pista 2</summary>

Para el recall, repite `cross_val_score` con `scoring="recall"` y promedia.
</details>

---
## 🏋️ Reto final: el umbral que le conviene al banco
Revisar una transacción sana cuesta `COSTO_FP` y dejar pasar un fraude cuesta `COSTO_FN`. Con `prob_log` e `y_test`:
1. `costos`: una Series con el costo total de cada umbral de `np.round(np.arange(0.05, 0.96, 0.05), 2)` (el umbral como índice). Costo total = `COSTO_FP` · FP + `COSTO_FN` · FN.
2. `umbral_optimo`: el umbral de menor costo.
3. `ahorro`: cuánto se ahorra usando `umbral_optimo` en lugar de 0.5.
4. `fig_costos, ax_costos`: una línea con el costo según el umbral, con una línea vertical `TINTA_2` en el umbral óptimo y el eje y en soles.

Escribe en una celda de texto por qué el umbral óptimo queda lejos de 0.5.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Recorre los umbrales con un bucle; para cada uno, predice con `prob_log >= u`, cuenta FP y FN y calcula el costo.
</details>

<details><summary>💡 Pista 2</summary>

Puedes reutilizar `confusion_matrix(y_test, prob_log >= u).ravel()`. Guarda los costos en una lista y conviértela en Series con `index=umbrales`. `costos.idxmin()` da el umbral óptimo.
</details>

---
## 🚀 Nivel pro (opcional): pesar más la clase rara
`LogisticRegression(max_iter=1000, class_weight="balanced")` le da más peso a los fraudes al entrenar, como si fueran tan frecuentes como las transacciones sanas. Entrena `modelo_bal` con los datos de entrenamiento y calcula `recall_bal` y `precision_bal` en prueba (umbral 0.5). Compara con `recall` y `precision`: ¿qué ganaste y qué perdiste? ¿Se parece a bajar el umbral?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar por qué un modelo con 97 % de accuracy puede no servir, y compararlo con un modelo tonto.
- [ ] Leer una matriz de confusión y nombrar sus cuatro celdas.
- [ ] Calcular precision, recall y F1 a mano y con scikit-learn, y explicar cuándo importa más cada una.
- [ ] Explicar qué muestra la curva ROC y qué significa un AUC de 0.5 y uno de 0.9.
- [ ] Explicar por qué la validación cruzada da una estimación más confiable y usar `StratifiedKFold`.
- [ ] Elegir un umbral según el costo de cada tipo de error.

**Próxima sesión (S20):** modelos más fuertes: overfitting, hiperparámetros, `GridSearchCV`, Random Forest y gradient boosting.